# Run-book 59-P2 — Window A into the new archive layout

**What this proves** (spec 59, `specs/59-imagery-archive-layout.md`):
- **AC 18** — S2 and S1 downloaded into ONE archive root land in two collection folders with two
  catalogs, and repeating the download fetches nothing.
- **AC 19** — adding a second processing of one acquisition (a CDSE granule) makes a default
  build refuse; `processing="latest"` builds, and the cube's provenance names only what it used.
- **AC 20** — a download killed mid-flight leaves nothing under the final file name, locally
  (a gate) and on blob (a recorded result: either answer is informative).
- **AC 21** — one S2 and one S1 cube look right in QGIS (your eyes).

**Window A:** grid cell `s2grid=4772924` — a ~5 km cell near Krems, Lower Austria
(~15.43 °E, 48.51 °N, north-west of Vienna). It sits entirely inside MGRS tile T33UWP, and only
relative orbit R122 passes over it (21 of 21 archive granules Apr–Sep 2018, one per date —
checked 2026-09-30). S2: June 2018. S1: 1–13 June 2018, one orbit track.

**Claude does not run this** (spec 24). You run it cell by cell and paste back the **Summary**
cell's output, step 4b's printed outcome, and any failing cell's message.

**Needs:**
- ~11 GB disk (estimate: S2 ≈ 6 granules × ~0.5 GB; S1 ≈ 2 scenes × 3.7 GB, the S1 figure
  measured in spec 58). Step 0 measures the real counts before anything downloads.
- Step 3b: a CDSE credentials JSON (S3 keys **and** Sentinel Hub client id/secret).
- Step 4b: write access to the blob container and a live `az login`.
- Time: step 1 ~5–15 min; steps 3c and 5 build datacubes (a few minutes each); the rest is short.
  All estimates.

## Before you start

**Kernel — which Python runs the cells.** Use **`.venv`**: the Python at `fsd/.venv/bin/python`
in the **main** checkout (Python 3.11). It is the only environment with fsd's dependencies.
- Open VS Code on the **`fsd/` folder** (the main checkout) — even when this notebook lives in a
  worktree under `fsd/.claude/worktrees/<name>/`. Then **Select Kernel** (top right) →
  **Python Environments** → **`.venv`**.
- Not listed (e.g. VS Code was opened on the worktree folder, which has no `.venv` of its own)?
  **Select Another Kernel… → Python Environments → Enter interpreter path…** (wording varies by
  VS Code version) and give the main checkout's `fsd/.venv/bin/python`.
- The **Setup** cell below checks this and stops with a message if the kernel is wrong.

**Code — which `fsd` gets imported.** fsd is installed from the *main* checkout, so inside a
worktree a plain `import fsd` would quietly run `main`'s code, not the branch under test. The
Setup cell puts **this notebook's own** `src/` first on Python's import path and checks it worked.
(This is what the old run-books did with `export PYTHONPATH=src` — the same thing, invisibly.)

**Stop / resume.** Interrupt the kernel (■) to stop. Re-running a cell resumes: downloads skip
files that are already complete. After a kernel restart, run **Setup** and **Settings** again,
then continue at any step — every step reads what it needs from disk.

**When you are done.** Paste back what the **Summary** cell says to paste (plus any failing cell's
message). Then **Restart & Clear All Outputs** and save: this file is tracked in git, and a test
refuses saved outputs.

In [ ]:
# Setup -- which Python, and which fsd. Nothing to change here.
import sys
from pathlib import Path

# 1. Which Python? It must be fsd's own virtual environment, fsd/.venv.
VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix} is not fsd's .venv. Pick the '.venv' kernel (see above).")
MAIN_CHECKOUT = VENV.parent              # the fsd checkout that owns the venv

# 2. Which fsd? The one next to this notebook: <checkout>/runbooks/<this> -> <checkout>/src.
#    Jupyter and VS Code start the kernel in the notebook's own folder.
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "fsd").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
import fsd

assert Path(fsd.__file__).is_relative_to(CHECKOUT / "src"), (
    f"fsd was already imported from {fsd.__file__}. Restart the kernel, then run this cell first.")
print("Python :", sys.executable)
print("fsd    :", fsd.__file__)

## Settings

Every input this run-book uses. Change them here and nowhere else.

In [ ]:
# Settings
import json
import math
import re
import shutil
import subprocess

import numpy as np
import pandas as pd
import rasterio

from fsd.catalog.catalog import TileCatalog, filter_by_properties
from fsd.sources import mpc
from fsd.workflows.create_datacube import run_create_datacube

OUT = MAIN_CHECKOUT / "tests" / "outputs" / "p59_p2"   # gitignored; outlives a worktree
ARCHIVE = OUT / "imagery"        # the ONE archive root that S2 and S1 share (spec 59 D2/D5)
ROI = CHECKOUT / "notebooks" / "shapefiles" / "s2grid=4772924.geojson"   # the Window A cell

S2, S1 = "sentinel-2-l2a", "sentinel-1-rtc"

S2_BANDS = ["B04", "B08", "SCL"]
S2_START = pd.Timestamp("2018-06-01", tz="UTC")
S2_END = pd.Timestamp("2018-07-01", tz="UTC")       # the end is exclusive: this keeps 30 June
S2_MAX_GRANULES = 15             # download() RAISES (does not trim) if more than this match

S1_BANDS = ["vv", "vh"]
S1_START = pd.Timestamp("2018-06-01", tz="UTC")
S1_END = pd.Timestamp("2018-06-13", tz="UTC")
S1_ORBIT = {"sat:orbit_state": "ascending", "sat:relative_orbit": 146}   # ONE track (spec 58 probe)
S1_MAX_SCENES = 3                # each scene is a ~3.7 GB whole-file copy, not a 5 km clip

CDSE_CREDENTIALS = Path.home() / "cdse_credentials.json"   # step 3b only

# Step 4: one small granule on one known date (S2A, 2018-06-05, R122/T33UWP). One day, because
# download() raises when more than max_tiles granules match.
KILL_START = pd.Timestamp("2018-06-05", tz="UTC")
KILL_END = pd.Timestamp("2018-06-06", tz="UTC")
KILL_LOCAL = OUT / "kill_local"
BLOB_ROOT = "abfss://data@strisewesteurope.dfs.core.windows.net/nsasiraj/demo-runs"   # step 4b


def read_catalog(collection):
    """The archive's catalog for one collection, as a GeoDataFrame."""
    return TileCatalog(str(ARCHIVE / collection / "catalog.parquet")).read()


def list_tifs(root):
    """{path relative to root: (size in bytes, modified time)} for every .tif under root."""
    return {str(p.relative_to(root)): (p.stat().st_size, p.stat().st_mtime)
            for p in Path(root).rglob("*.tif")}

## Step 0 — plan: what would step 1 download? (no transfer)

**What it does:** asks MPC's catalog which S2 granules and S1 scenes match, and estimates the
bytes. Nothing is downloaded.
**PASS if:** the counts fit under the limits step 1 passes to `download()` (it refuses more), and
the disk has room for the estimate plus 5 GB.
**Expect:** ~6 S2 granules (R122 every 5 days), 2 S1 scenes.

In [ ]:
# Step 0 -- discovery only (a catalog search), no bytes transferred.
s2_found = mpc.query_catalog(str(ROI), S2_START, S2_END, collection=S2)
s1_found = filter_by_properties(mpc.query_catalog(str(ROI), S1_START, S1_END, collection=S1),
                                S1_ORBIT)
print(s2_found[["id", "cloud_cover", "processing_version"]].to_string(), "\n")
print(s1_found[["id"]].to_string(), "\n")

# Per-file sizes are estimates: ~0.5 GB for an S2 granule's B04+B08+SCL, 3.7 GB for an S1 scene.
estimate_gb = len(s2_found) * 0.5 + len(s1_found) * 3.7
free_gb = shutil.disk_usage(MAIN_CHECKOUT).free / 1e9
print(f"S2 granules: {len(s2_found)}   S1 scenes: {len(s1_found)}   "
      f"estimate: {estimate_gb:.1f} GB   free: {free_gb:.1f} GB")

assert len(s2_found) <= S2_MAX_GRANULES, f"{len(s2_found)} S2 granules > {S2_MAX_GRANULES}"
assert len(s1_found) <= S1_MAX_SCENES, f"{len(s1_found)} S1 scenes > {S1_MAX_SCENES}"
assert free_gb > estimate_gb + 5, f"only {free_gb:.1f} GB free for ~{estimate_gb:.1f} GB"

## Step 1 — download S2 and S1 into ONE archive root (AC 18, spec 59 D2/D5)

**What it does:** two `fsd.download` calls with the same `dst_folderpath`. Each prints its own
progress. Interrupting and re-running this cell resumes (finished files are skipped).
**PASS if:** the archive holds exactly two folders, `sentinel-1-rtc/` and `sentinel-2-l2a/`,
each with its own `catalog.parquet` and at least one row; granule folders follow
`{collection}/YYYY/MM/DD/{granule name}/`; S2 rows carry MPC's processing version (expect
`02.12`) and S1 rows carry none.
**Record:** the size and minutes it prints — they size the full Austria re-download.

To start over from an empty archive, run the optional cell right after this one first.

In [ ]:
# Step 1 -- download. download_window_a() is reused by step 2.
def download_window_a():
    """Step 1's two downloads: S2 and S1 into the one shared ARCHIVE."""
    fsd.download(roi=str(ROI), startdate=S2_START, enddate=S2_END, bands=S2_BANDS,
                 dst_folderpath=str(ARCHIVE), collection=S2, max_tiles=S2_MAX_GRANULES)
    fsd.download(roi=str(ROI), startdate=S1_START, enddate=S1_END, bands=S1_BANDS,
                 dst_folderpath=str(ARCHIVE), collection=S1, max_tiles=S1_MAX_SCENES,
                 properties_filter=S1_ORBIT)


started = pd.Timestamp.now()
download_window_a()
print(f"step 1 took {(pd.Timestamp.now() - started).total_seconds() / 60:.1f} min")

In [ ]:
# OPTIONAL -- start step 1 from an EMPTY archive. Deletes everything under ARCHIVE.
# Set to True and run, then run step 1 again.
START_OVER = False
if START_OVER:
    shutil.rmtree(ARCHIVE, ignore_errors=True)
    print("deleted", ARCHIVE)

In [ ]:
# Step 1 -- check.
s2, s1 = read_catalog(S2), read_catalog(S1)
folders = sorted(p.name for p in ARCHIVE.iterdir() if p.is_dir())
tifs = list_tifs(ARCHIVE)
s2_granule = Path(s2["local_folderpath"].iloc[0]).relative_to(ARCHIVE)
s1_granule = Path(s1["local_folderpath"].iloc[0]).relative_to(ARCHIVE)

print("collection folders :", folders)
print("S2 rows / S1 rows  :", len(s2), "/", len(s1))
print("S2 granule folder  :", s2_granule)
print("S1 granule folder  :", s1_granule)
print("S2 versions        :", sorted(set(s2["processing_version"].astype(str))))
print("S1 versions        :", sorted(set(s1["processing_version"].astype(str))))
print(f"archive size       : {sum(size for size, _ in tifs.values()) / 1e9:.2f} GB in {len(tifs)} files")

assert folders == [S1, S2], f"expected one folder per collection, found {folders}"
assert (ARCHIVE / S2 / "catalog.parquet").exists() and (ARCHIVE / S1 / "catalog.parquet").exists()
assert len(s2) > 0 and len(s1) > 0, "each catalog must have at least one row"
# {collection}/YYYY/MM/DD/{granule name}: five parts, the first the collection
assert s2_granule.parts[0] == S2 and len(s2_granule.parts) == 5, f"S2 layout: {s2_granule}"
assert s1_granule.parts[0] == S1 and len(s1_granule.parts) == 5, f"S1 layout: {s1_granule}"
assert s1["processing_version"].isna().all(), "S1 RTC publishes no processing version"

## Step 2 — the identical download again fetches nothing (AC 18)

**What it does:** notes every `.tif` (size + modified time) and each catalog's row count, runs
step 1's downloads again, and compares.
**PASS if:** no `.tif` was created or rewritten and the row counts are unchanged. The evidence is
the files themselves, not what the download printed.
**Time:** seconds to a minute.

In [ ]:
# Step 2 -- repeat step 1 and compare the files before and after.
tifs_before = list_tifs(ARCHIVE)
rows_before = {S2: len(read_catalog(S2)), S1: len(read_catalog(S1))}

download_window_a()

tifs_after = list_tifs(ARCHIVE)
rows_after = {S2: len(read_catalog(S2)), S1: len(read_catalog(S1))}
changed = sorted(p for p in tifs_after if tifs_before.get(p) != tifs_after[p])
print(f"{len(tifs_after)} .tif files, {len(changed)} new or rewritten; rows {rows_before} -> {rows_after}")

assert changed == [], (f"{len(changed)} files were (re)written, e.g. {changed[:3]}: the skip "
                       "check is not seeing the published files (a D2/D10 regression)")
assert rows_after == rows_before, "the catalogs gained or lost rows"

## Step 3a — pick an acquisition for CDSE to add (no network)

**What it does:** lists the S2 acquisitions already in the archive, least cloudy first.
**PASS if:** every one carries MPC's pre-05.00 processing (so a CDSE copy, reprocessed as 05.00,
is a *different* processing of the same acquisition).
**Then:** copy the first date into `CDSE_DAY` in step 3b.

In [ ]:
# Step 3a -- candidates for step 3b.
s2 = read_catalog(S2)
candidates = s2.sort_values("cloud_cover")[
    ["timestamp", "acquisition_key", "processing_version", "source", "cloud_cover"]].head(6)
display(candidates)

assert (candidates["processing_version"].astype(str) < "05.00").all(), (
    "expected MPC's original (pre-05.00) processing on every candidate")

## Step 3b — add ONE CDSE granule for that acquisition (AC 19, spec 59 D3/D4/D7)

**What it does:** `fsd.download(source="cdse", processing="latest", max_tiles=1)` over that one
day. CDSE serves the reprocessed baseline (05.00), MPC holds the original (02.12), so the two
canonical granule names differ: a **separate folder and row** (D3) with the **same acquisition
key** (D4). Only the local CDSE path is exercised here.
**PASS if:** the archive now has exactly one CDSE row, and that acquisition has two rows — one
per source — in two different folders.
**Look for** this line in the download's output (D7's warning; not asserted, since reading
printed text would need capture plumbing): `[download] archive now holds >1 processing …`.
**If it fails** with "one row, sources cdse,mpc": CDSE served the *same* processing MPC holds (a
merge, not a duplicate). Pick the next candidate from 3a. Before re-running, delete that CDSE
granule's folder or the row stays merged.
**Time:** one granule plus a jp2→COG conversion — a few minutes.

In [ ]:
# Step 3b -- add the CDSE copy.
from fsd.sources.cdse import CdseCredentials

CDSE_DAY = None     # <- set to a date from step 3a, e.g. pd.Timestamp("2018-06-20", tz="UTC")
assert CDSE_DAY is not None, "set CDSE_DAY to a date from step 3a's table first"

fsd.download(roi=str(ROI), startdate=CDSE_DAY, enddate=CDSE_DAY + pd.Timedelta(days=1),
             bands=S2_BANDS, dst_folderpath=str(ARCHIVE), collection=S2,
             source="cdse", creds=CdseCredentials.from_json(str(CDSE_CREDENTIALS)),
             processing="latest", max_tiles=1)

In [ ]:
# Step 3b -- check (reads the catalog, so it works after a restart).
s2 = read_catalog(S2)
from_cdse = s2[s2["source"].str.contains("cdse")]
assert len(from_cdse) == 1, f"expected exactly 1 row from CDSE, found {len(from_cdse)}"
same_acquisition = s2[s2["acquisition_key"] == from_cdse["acquisition_key"].iloc[0]]
display(same_acquisition[["id", "source", "processing_version", "local_folderpath"]])

assert len(same_acquisition) == 2, (
    f"expected MPC's and CDSE's copies as TWO rows, found {len(same_acquisition)} "
    f"(sources: {list(same_acquisition['source'])})")
assert set(same_acquisition["source"]) == {"mpc", "cdse"}
assert same_acquisition["local_folderpath"].nunique() == 2, "two processings need two folders (D3)"

## Step 3c — build over the ambiguity (AC 19, spec 59 D6/D9)

**What it does:** builds the Window A S2 cube twice with the local runner:
(i) `processing=None` (the build default) — must **refuse**, because one acquisition now has two
processings; (ii) `processing="latest"` — must build, and its provenance must list the CDSE copy
of that acquisition and **not** the superseded MPC copy.
**PASS if:** (i) raises a `ValueError` whose message names the acquisition and the
`processing='latest'` fix; (ii) builds, and the provenance checks below hold.
**Paste** the refusal message it prints — it is what a user will read.
**Time:** a few minutes for the build.

In [ ]:
# Step 3c -- a helper to build one datacube over the ROI (step 5 reuses it).
def build_cube(collection, bands, start, end, processing, run_folder):
    """Build the ROI's datacube with the local runner; return its datacube.npy path."""
    run_create_datacube(
        catalog_filepath=str(ARCHIVE / collection / "catalog.parquet"),
        timestamp_col="timestamp", shapefilepath=str(ROI), id_col="id",
        run_folderpath=str(run_folder / "run"), csv_filepath=str(run_folder / "input.csv"),
        startdate=start, enddate=end, bands=bands, mosaic_days=10,
        label_col=None, cores=1, collection=collection, runner="local", processing=processing)
    return pd.read_csv(run_folder / "input.csv")["datacube_filepath"].iloc[0]   # one ROI -> one row


s2 = read_catalog(S2)
versions_per_acquisition = s2.groupby("acquisition_key")["id"].nunique()
ambiguous = versions_per_acquisition[versions_per_acquisition > 1].index.tolist()
print("acquisitions held in more than one processing:", ambiguous)
assert len(ambiguous) == 1, "step 3b should have left exactly one such acquisition"
ambiguous_key = ambiguous[0]

In [ ]:
# Step 3c (i) -- processing=None must refuse.
try:
    build_cube(S2, S2_BANDS, S2_START, S2_END, processing=None, run_folder=OUT / "build_none")
except ValueError as refusal:
    message = str(refusal)
else:
    raise AssertionError("processing=None built a cube; spec 59 D6 says it must refuse")

print(message)
assert ambiguous_key in message, "the refusal must name the ambiguous acquisition"
assert "processing='latest'" in message, "the refusal must name the fix"

In [ ]:
# Step 3c (ii) -- processing="latest" builds; the provenance says what it used.
cube_path = build_cube(S2, S2_BANDS, S2_START, S2_END, processing="latest",
                       run_folder=OUT / "build_latest")
metadata = np.load(Path(cube_path).with_name("metadata.pickle.npy"), allow_pickle=True).item()
used = metadata["provenance"][S2]
print("cube:", cube_path)
print("used versions:", used["processing_version"], " sources:", used["source"],
      f" granules: {len(used['ids'])}")

rows = s2[s2["acquisition_key"] == ambiguous_key]
cdse_copy = rows[rows["source"].str.contains("cdse")]["id"].iloc[0]
mpc_copy = rows[~rows["source"].str.contains("cdse")]["id"].iloc[0]
assert cdse_copy in used["ids"], "'latest' must use the newer (CDSE, 05.00) copy"
assert mpc_copy not in used["ids"], "the superseded MPC copy of that acquisition must not be used"
assert set(used["processing_version"]) <= set(s2["processing_version"].astype(str))

## Step 4a — kill a local download between transfer and publish (AC 20, spec 59 D10)

**What it does:** a **child process** runs a real one-granule download (B04, 2018-06-05) into
`KILL_LOCAL`, with the stamp step replaced by an instant exit — it dies right after the bytes
land, before they are stamped and published. (A child, because `os._exit` here would kill this
notebook's kernel.) The next cell then re-runs the same download normally.
**PASS if:** after the kill there is **no** `.tif` under its final name, only a `.tif.stage`
file; after the re-run there is exactly one `.tif`, stamped with S2's reflectance scale 1e-4, and
no `.stage` left.

In [ ]:
# Step 4a -- the child: a real download that dies right after the transfer.
CHILD_LOCAL = """
import os, sys
src, roi, dst, start, end = sys.argv[1:]
sys.path.insert(0, src)                 # the same fsd this notebook runs
import fsd
from fsd.sources import mpc

def die(*args, **kwargs):
    print("[child] bytes are on disk; dying before stamp + publish", flush=True)
    os._exit(137)                       # like a SIGKILL: no cleanup code runs

mpc.stamp_or_reencode = die             # the step right after the transfer
fsd.download(roi=roi, startdate=start, enddate=end, bands=["B04"], dst_folderpath=dst, max_tiles=1)
"""
shutil.rmtree(KILL_LOCAL, ignore_errors=True)
child = subprocess.run(
    [sys.executable, "-c", CHILD_LOCAL, str(CHECKOUT / "src"), str(ROI), str(KILL_LOCAL),
     KILL_START.isoformat(), KILL_END.isoformat()],
    capture_output=True, text=True)
print(child.stdout[-2000:], child.stderr[-2000:])

published = sorted(KILL_LOCAL.rglob("*.tif"))
staged = sorted(KILL_LOCAL.rglob("*.tif.stage"))
print("exit code:", child.returncode, "| published:", published, "| staged:", staged)
assert child.returncode == 137, "the child should have died at the stamp step (exit code 137)"
assert published == [], "a killed download must leave NO file under the final name"
assert len(staged) == 1, "the transferred bytes should be waiting in one .tif.stage file"

In [ ]:
# Step 4a -- re-run normally: it must transfer again and publish a stamped file.
fsd.download(roi=str(ROI), startdate=KILL_START, enddate=KILL_END, bands=["B04"],
             dst_folderpath=str(KILL_LOCAL), max_tiles=1)

published = sorted(KILL_LOCAL.rglob("*.tif"))
staged = sorted(KILL_LOCAL.rglob("*.tif.stage"))
assert len(published) == 1 and staged == [], f"published={published} staged={staged}"
with rasterio.open(published[0]) as f:
    scale = f.scales[0]
print("published:", published[0].relative_to(KILL_LOCAL), "| scale:", scale)
assert math.isclose(scale, 1e-4), f"expected S2's reflectance scale 1e-4, found {scale}"

## Step 4b — kill a blob upload mid-way (AC 20: a recorded result, not a gate)

**What it does:** a child process downloads the same granule to a fresh
`…/demo-runs/p59_kill_<UTC time>/` folder on blob. For a blob destination fsd downloads to a local
scratch file, stamps it, then **uploads** it; the child is killed `KILL_DELAY_S` seconds into
that upload. This cell then lists what is on blob under that folder.

**Outcomes** (printed as `OUTCOME:`):
- *NO blob under the final name* — a killed upload publishes nothing. **PASS.**
- *a PARTIAL blob is visible* — **PASS as a result**, and a GitHub issue must be filed for D10's
  remote half (`gh issue create`, quoting this output). Not a silent pass.
- *upload finished before the kill* — inconclusive: set `KILL_DELAY_S` to 0.5, then 0.2, re-run.
- *the upload failed on its own* — nothing tested; paste the child's output.

**Paste** this cell's whole output. **Clean up** with the next cell afterwards.

In [ ]:
# Step 4b -- kill the upload to blob, then look at what is there.
from datetime import datetime, timezone

from fsd.storage import fs
from fsd.storage.azure import configure_storage

KILL_DELAY_S = 1.5      # seconds into the upload; lower it if the upload wins the race

CHILD_BLOB = """
import os, sys, threading, time
src, roi, dst, start, end, delay = sys.argv[1:]
sys.path.insert(0, src)
import fsd
from fsd.sources import mpc

real_put = mpc.fs.put

def put_then_die(local_path, remote_path, **kwargs):
    print(f"[child] upload starts: {os.path.getsize(local_path)} bytes -> {remote_path}", flush=True)
    def die():
        time.sleep(float(delay))
        print(f"[child] {delay} s into the upload: killing", flush=True)
        os._exit(137)
    threading.Thread(target=die, daemon=True).start()
    return real_put(local_path, remote_path, **kwargs)

mpc.fs.put = put_then_die               # the upload of the finished local file to blob
fsd.download(roi=roi, startdate=start, enddate=end, bands=["B04"], dst_folderpath=dst,
             max_tiles=1, storage="azure")
"""
prefix = f"{BLOB_ROOT}/p59_kill_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"
child = subprocess.run(
    [sys.executable, "-c", CHILD_BLOB, str(CHECKOUT / "src"), str(ROI), prefix,
     KILL_START.isoformat(), KILL_END.isoformat(), str(KILL_DELAY_S)],
    capture_output=True, text=True)
print(child.stdout[-2000:], child.stderr[-2000:])

started = re.search(r"upload starts: (\d+) bytes", child.stdout)
assert started, "the child never reached the upload: nothing was tested (read its output above)"
local_size = int(started.group(1))

configure_storage("azure")              # list blobs as you (az login), not anonymously
try:
    on_blob = fs.find_sizes(prefix)     # {path: size}; raises if the folder does not exist
except FileNotFoundError:
    on_blob = {}
tifs = {path: size for path, size in on_blob.items() if path.endswith(".tif")}
print("exit code:", child.returncode, "| local file:", local_size, "bytes")
print("everything under the prefix:", json.dumps(on_blob, indent=2))

if tifs and all(size == local_size for size in tifs.values()):
    outcome = "upload finished before the kill -- inconclusive; lower KILL_DELAY_S and re-run"
elif child.returncode != 137:
    outcome = f"the upload failed on its own (exit {child.returncode}) -- nothing tested"
elif not tifs:
    outcome = "NO blob under the final name -- a killed upload publishes nothing (PASS)"
else:
    outcome = "a PARTIAL blob is visible under the final name (PASS as a result; file an issue)"
print("OUTCOME:", outcome)

In [ ]:
# Step 4b -- clean up: lists the test folders; deletes them only when DELETE is True.
DELETE = False
leftovers = sorted(Path(p).name for p in fs.glob(f"{BLOB_ROOT}/p59_kill_*"))
print("test folders on blob:", leftovers)
if DELETE:
    for name in leftovers:
        fs.rm(f"{BLOB_ROOT}/{name}", recursive=True)
    print("deleted", len(leftovers))

## Step 5 — one S2 and one S1 cube, for QGIS (AC 21)

**What it does:** reuses step 3c's S2 cube (`processing="latest"`, so the CDSE granule is in
play) and builds the S1 cube (default `processing=None`: S1 has no duplicate processings). For
each, writes the fullest timestamp as a GeoTIFF under `OUT/qgis/`, with the collection's own
nodata value (0 for S2, **−32768** for S1).
**PASS if:** both cubes build with less than 90 % nodata **and your eyes agree** in QGIS: open both
over a basemap — right place (near Krems, ~15.43 °E 48.51 °N), no shift or flip, S2 looks like
terrain, S1 vv/vh look like plausible backscatter (linear values, mostly 0–0.5; stretch to
~2–98 %). Tell Claude yes or no.

In [ ]:
# Step 5 -- write the fullest timestamp of each cube as a GeoTIFF.
from fsd import collections as fsd_collections


def save_fullest_timestamp(cube_path, nodata, dst):
    """Write the cube's fullest timestamp (all bands) to `dst`; return what QGIS will show."""
    cube = np.load(cube_path)                     # (timestamps, height, width, bands)
    metadata = np.load(Path(cube_path).with_name("metadata.pickle.npy"), allow_pickle=True).item()
    filled = [(cube[t] != nodata).mean() for t in range(cube.shape[0])]
    t = int(np.argmax(filled))
    profile = metadata["geotiff_metadata"]        # the grid the cube was built on
    with rasterio.open(dst, "w", driver="GTiff", count=cube.shape[3], height=cube.shape[1],
                       width=cube.shape[2], dtype=str(cube.dtype), crs=profile["crs"],
                       transform=profile["transform"], nodata=nodata) as f:
        f.write(np.moveaxis(cube[t], -1, 0))      # rasterio wants (bands, height, width)
        f.descriptions = tuple(metadata["bands"])
    return {"shape": cube.shape, "bands": metadata["bands"], "timestamp_written": t,
            "nodata_fraction": round(float((cube == nodata).mean()), 3), "geotiff": str(dst)}


(OUT / "qgis").mkdir(parents=True, exist_ok=True)
s2_cube = build_cube(S2, S2_BANDS, S2_START, S2_END, processing="latest",
                     run_folder=OUT / "build_latest")          # step 3c's cube; not rebuilt
s1_cube = build_cube(S1, S1_BANDS, S1_START, S1_END, processing=None,
                     run_folder=OUT / "build_s1")
s2_info = save_fullest_timestamp(s2_cube, fsd_collections.get(S2).nodata, OUT / "qgis" / "s2.tif")
s1_info = save_fullest_timestamp(s1_cube, fsd_collections.get(S1).nodata, OUT / "qgis" / "s1.tif")
print("S2:", s2_info)
print("S1:", s1_info)

assert s2_info["nodata_fraction"] < 0.9, "the S2 cube is (almost) all nodata"
assert s1_info["nodata_fraction"] < 0.9, "the S1 cube is (almost) all nodata"

## Summary — paste this output back

Also paste step 4b's `OUTCOME:` line and your QGIS yes/no. Re-reads everything from disk.

In [ ]:
s2, s1 = read_catalog(S2), read_catalog(S1)
tifs = list_tifs(ARCHIVE)
summary = {
    "archive_GB": round(sum(size for size, _ in tifs.values()) / 1e9, 2),
    "archive_tif_files": len(tifs),
    "s2_rows": len(s2),
    "s1_rows": len(s1),
    "s2_versions_by_source": sorted(set(zip(s2["source"], s2["processing_version"].astype(str)))),
    "kill_local_files": sorted(str(p.relative_to(KILL_LOCAL)) for p in KILL_LOCAL.rglob("*.tif*")),
    "qgis_files": sorted(p.name for p in (OUT / "qgis").glob("*.tif")),
}
print(json.dumps(summary, indent=2, default=str))